# Disponibilidad sintetica

Este cuaderno revisa disponibilidad y latencia de tests sinteticos para decidir como mostrar disponibilidad en el dashboard.

In [ ]:
import sys
from pathlib import Path
import pandas as pd

NOTEBOOK_CWD = Path.cwd()
candidate_roots = [
    NOTEBOOK_CWD,
    NOTEBOOK_CWD.parent if NOTEBOOK_CWD.name == 'notebooks' else NOTEBOOK_CWD,
    NOTEBOOK_CWD / 'observability' / 'd365-fo-observability',
]
PROJECT_ROOT = None
for candidate in candidate_roots:
    if (candidate / 'src' / 'kql_runner.py').exists() and (candidate / 'queries').exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError('No se pudo localizar observability/d365-fo-observability desde el directorio actual')
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from kql_runner import build_client, load_config, load_kql, metric_snapshot, plot_bar, run_kql, set_analyst_theme

config = load_config()
client = build_client(config=config)
AVAILABILITY_DAYS = max(config['query_days'], 30)
set_analyst_theme()

In [ ]:
df_availability = run_kql(client, load_kql('60_availability.kql'), days=AVAILABILITY_DAYS, name='Availability', config=config)
display(df_availability)
if not df_availability.empty:
    snapshot = metric_snapshot({
        'Numero de tests': len(df_availability),
        'Peor disponibilidad %': round(pd.to_numeric(df_availability['AvailabilityPercent'], errors='coerce').min(), 3),
        'Mejor disponibilidad %': round(pd.to_numeric(df_availability['AvailabilityPercent'], errors='coerce').max(), 3),
        'Latencia media maxima (s)': round(pd.to_numeric(df_availability['AvgDurationSeconds'], errors='coerce').max(), 2),
    })
    display(snapshot)

In [ ]:
plot_bar(df_availability, 'name', 'AvailabilityPercent', 'Disponibilidad por test', top_n=20, ascending=True)
plot_bar(df_availability, 'name', 'AvgDurationSeconds', 'Latencia media por test', top_n=20)

## Decision de dashboard

- Si hay varios tests, un ranking por disponibilidad es intuitivo y directo.
- Si solo hay un test, suele bastar una tarjeta KPI con posibilidad de drill-through.
- La latencia media ayuda a detectar degradacion antes del fallo, por eso conviene mostrarla cerca del KPI de disponibilidad.